In [1]:
import os
import sys
import datetime

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

26/09/24 21:53:49 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/09/24 21:53:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 21:53:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


26/09/24 21:53:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/24 21:53:49 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/24 21:53:49 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


### Generate updated integrated taxitrip table

In [2]:
import time
from src.monitoring.storage_utils import get_dir_size_bytes
from src.monitoring.pipeline_monitor import PipelineRunMetadata, new_run_id, record_pipeline_run
from src.integration.integrated_taxi_trips import build_integrated_taxi_trips_incremental, GOLD_PATH
from src.common.delta_writer import write_delta

run_id = new_run_id()
started_at = datetime.datetime.now(datetime.timezone.utc)
gold_storage_before = get_dir_size_bytes(GOLD_PATH)
t0 = time.time()
status, error_message, n_new = "SUCCESS", "", 0

try:
    new_gold_df, n_new = build_integrated_taxi_trips_incremental(spark)
    if new_gold_df is None:
        print("Gold: no new trips to add, skipping rebuild")
        status = "SKIPPED"
    else:
        write_delta(new_gold_df, GOLD_PATH, mode="append", partition_by=["pickup_year", "pickup_month"])
        print(f"Gold: appended {n_new} new trip rows incrementally")
except Exception as exc:
    status, error_message = "FAILED", str(exc)
    raise
finally:
    gold_build_time = time.time() - t0
    gold_storage_overhead = get_dir_size_bytes(GOLD_PATH) - gold_storage_before
    record_pipeline_run(PipelineRunMetadata(
        run_id=run_id, pipeline_name="gold_build", dataset_name="integrated_taxi_trips",
        schema_version="v1.0", status=status, records_processed=n_new, records_inserted=n_new,
        storage_overhead_bytes=gold_storage_overhead, error_message=error_message,
        started_at=started_at.isoformat(), ended_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        execution_time_seconds=round(gold_build_time, 3),
    ))
    print(f"Gold build time: {gold_build_time:.2f}s | storage added: {gold_storage_overhead/1e6:.3f} MB")

/Users/thilini/.pyenv/versions/3.12.0/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
26/09/24 21:54:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Gold: appended 293670 new trip rows incrementally
Gold build time: 19.49s | storage added: 25.799 MB


### Generate refresh product tables

In [5]:
import os
import datetime
from pyspark.sql import functions as F
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
from src.common.delta_writer import table_exists
from src.monitoring.pipeline_monitor import PipelineTimer, PipelineRunMetadata, new_run_id, record_pipeline_run

PRODUCTS_DIR = os.path.join(PROJECT_ROOT, "data", "delta", "gold", "data_products")


def add_audit_metadata(df, schema_version="v1.0"):
    now = datetime.datetime.now(datetime.timezone.utc)
    return df \
        .withColumn("_created_at", F.lit(now.strftime("%Y-%m-%d %H:%M:%S"))) \
        .withColumn("_updated_at", F.lit(now.strftime("%Y-%m-%d %H:%M:%S"))) \
        .withColumn("_source_version", F.lit("integrated_taxi_trips_v1")) \
        .withColumn("_schema_version", F.lit(schema_version))


def _merge_product(product_name, recompute_pdf, key_columns):
    product_path = os.path.join(PRODUCTS_DIR, product_name)
    if not table_exists(product_path):
        write_deltalake(product_path, recompute_pdf, mode="overwrite")
        print(f"{product_name}: created (first build), {len(recompute_pdf)} rows")
        return

    predicate = " AND ".join(f"target.{c} = source.{c}" for c in key_columns)
    dt = DeltaTable(product_path)
    (
        dt.merge(
            source=recompute_pdf,
            predicate=predicate,
            source_alias="source",
            target_alias="target",
        )
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )
    print(f"{product_name}: merged {len(recompute_pdf)} affected group(s)")


def _pm25_bucket_col(df):
    return df.withColumn(
        "pm25_level",
        F.when(F.greatest(F.col("air_quality_pm25"), F.lit(0.0)) <= 9.0, "1: Good (0.0-9.0)")
        .when(F.col("air_quality_pm25") <= 35.4, "2: Moderate (9.1-35.4)")
        .when(F.col("air_quality_pm25") <= 55.4, "3: Unhealthy for Sensitive Groups (35.5-55.4)")
        .when(F.col("air_quality_pm25") <= 125.4, "4: Unhealthy (55.5-125.4)")
        .otherwise("5: Very Unhealthy / Hazardous"),
    )


def refresh_products_incremental(spark, new_gold_df):
    run_id = new_run_id()
    started_at = datetime.datetime.now(datetime.timezone.utc)
    t0 = time.perf_counter()
    status, error_message = "SUCCESS", ""
    records_processed = 0
    records_inserted = 0

    try:
        if new_gold_df is None:
            print("No new Gold rows - all products already up to date, skipping.")
            status = "SKIPPED"
        else:
            gold = spark.read.parquet(GOLD_PATH).filter(F.col("is_valid")).cache()
            new_with_bucket = _pm25_bucket_col(new_gold_df)

            touched = new_with_bucket.agg(
                F.collect_set("event_date").alias("dates"),
                F.collect_set("pickup_zone").alias("zones"),
                F.collect_set("weather_condition_code").alias("codes"),
                F.collect_set("pm25_level").alias("buckets"),
            ).collect()[0]

            touched_dates = [d for d in touched["dates"] if d is not None]
            touched_zones = [z for z in touched["zones"] if z is not None]
            touched_codes = [c for c in touched["codes"] if c is not None]
            touched_buckets = [b for b in touched["buckets"] if b is not None]

            records_processed = new_gold_df.count()

            # 1. Daily Mobility Summary
            if touched_dates:
                recompute = gold.filter(F.col("event_date").isin(touched_dates)).groupBy(
                    "event_date", "pickup_borough"
                ).agg(
                    F.count("*").alias("total_trips"),
                    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
                    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
                    F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
                )
                pdf = add_audit_metadata(recompute).toPandas()
                records_inserted += len(pdf)
                _merge_product("daily_mobility_summary", pdf, key_columns=["event_date", "pickup_borough"])
            else:
                print("daily_mobility_summary: no touched dates, skipping")

            # 2. Taxi Zone Statistics
            if touched_zones:
                recompute = gold.filter(F.col("pickup_zone").isin(touched_zones)).groupBy(
                    "pickup_zone", "pickup_borough"
                ).agg(
                    F.count("*").alias("total_pickups"),
                    F.round(F.avg("fare_amount"), 2).alias("avg_pickup_fare"),
                    F.round(F.avg("trip_distance"), 2).alias("avg_pickup_distance"),
                )
                pdf = add_audit_metadata(recompute).toPandas()
                records_inserted += len(pdf)
                _merge_product("taxi_zone_statistics", pdf, key_columns=["pickup_zone", "pickup_borough"])
            else:
                print("taxi_zone_statistics: no touched zones, skipping")

            # 3. Weather Impact Summary
            if touched_codes:
                recompute = gold.filter(F.col("weather_condition_code").isin(touched_codes)).groupBy(
                    "weather_condition_code"
                ).agg(
                    F.count("*").alias("total_trips"),
                    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
                    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
                    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                )
                pdf = add_audit_metadata(recompute).toPandas()
                records_inserted += len(pdf)
                _merge_product("weather_impact_summary", pdf, key_columns=["weather_condition_code"])
            else:
                print("weather_impact_summary: no touched condition codes, skipping")

            # 4. Air Quality Impact Summary
            if touched_buckets:
                with_bucket = _pm25_bucket_col(gold)
                recompute = with_bucket.filter(F.col("pm25_level").isin(touched_buckets)).groupBy(
                    "pm25_level"
                ).agg(
                    F.count("*").alias("total_trips"),
                    F.round(F.avg("air_quality_pm25"), 2).alias("avg_pm25"),
                    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
                )
                pdf = add_audit_metadata(recompute).toPandas()
                records_inserted += len(pdf)
                _merge_product("air_quality_impact_summary", pdf, key_columns=["pm25_level"])
            else:
                print("air_quality_impact_summary: no touched pm25 buckets, skipping")

    except Exception as exc:
        status, error_message = "FAILED", str(exc)
        raise
    finally:
        elapsed = time.perf_counter() - t0
        ended_at = datetime.datetime.now(datetime.timezone.utc)
        record_pipeline_run(PipelineRunMetadata(
            run_id=run_id,
            pipeline_name="product_refresh",
            dataset_name="data_products",
            schema_version="v1.0",
            status=status,
            records_processed=records_processed,
            records_inserted=records_inserted,
            records_rejected=0,
            error_message=error_message,
            started_at=started_at.isoformat(),
            ended_at=ended_at.isoformat(),
            execution_time_seconds=round(elapsed, 3),
        ))

In [6]:
import time
from src.monitoring.storage_utils import get_dir_size_bytes

products_storage_before = get_dir_size_bytes(PRODUCTS_DIR)
t0 = time.time()
refresh_products_incremental(spark, new_gold_df)
incremental_time = time.time() - t0
products_storage_overhead = get_dir_size_bytes(PRODUCTS_DIR) - products_storage_before
print(f"Incremental product refresh: {incremental_time:.2f}s | storage added: {products_storage_overhead/1e6:.3f} MB")

26/09/24 22:03:28 WARN CacheManager: Asked to cache already cached data.
26/09/24 22:03:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/24 22:03:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but r

daily_mobility_summary: merged 108 affected group(s)
taxi_zone_statistics: merged 245 affected group(s)
weather_impact_summary: merged 8 affected group(s)
air_quality_impact_summary: merged 2 affected group(s)
Incremental product refresh: 13.37s | storage added: 0.054 MB


### Final evaluation

In [7]:
from src.ingestion.metadata import read_ingestion_runs
from src.monitoring.pipeline_monitor import read_pipeline_runs, PipelineRunMetadata, new_run_id, record_pipeline_run
from src.ingestion.metadata import IngestionRunMetadata, save_metadata, new_run_id as new_ingest_run_id
import time

ingestion_df = read_ingestion_runs()
pipeline_df = read_pipeline_runs()

latest_ingest = ingestion_df[ingestion_df["source_file"].str.contains("update", case=False, na=False)].sort_values("started_at").iloc[-1]
latest_gold = pipeline_df[pipeline_df["pipeline_name"] == "gold_build"].sort_values("started_at").iloc[-1]
latest_product = pipeline_df[pipeline_df["pipeline_name"] == "product_refresh"].sort_values("started_at").iloc[-1]

def benchmark_monitoring_write(kind="pipeline"):
    t0 = time.perf_counter()
    if kind == "pipeline":
        record_pipeline_run(PipelineRunMetadata(run_id=new_run_id(), pipeline_name="benchmark",
                                                  dataset_name="benchmark", schema_version="v1.0", status="SUCCESS"))
    else:
        save_metadata(IngestionRunMetadata(run_id=new_ingest_run_id(), dataset_name="benchmark",
                                            source_file="benchmark", format="parquet",
                                            schema_version="v1.0", status="SUCCESS"))
    return time.perf_counter() - t0

print("=" * 70)
print("METHOD 1: Incremental Update (ingestion + gold build)")
m1_time = latest_ingest["execution_time_seconds"] + latest_gold["execution_time_seconds"]
m1_storage = latest_ingest["storage_overhead_bytes"] + latest_gold["storage_overhead_bytes"]
m1_monitor = benchmark_monitoring_write("ingestion") + benchmark_monitoring_write("pipeline")
print(f"  Time:                {m1_time:.2f}s")
print(f"  Storage overhead:    {m1_storage/1e6:+.3f} MB")
print(f"  Monitoring overhead: {m1_monitor:.3f}s")

print("\nMETHOD 2: Refresh Product Tables")
m2_time = latest_product["execution_time_seconds"]
m2_storage = latest_product["storage_overhead_bytes"]
m2_monitor = benchmark_monitoring_write("pipeline")
print(f"  Time:                {m2_time:.2f}s")
print(f"  Storage overhead:    {m2_storage/1e6:+.3f} MB")
print(f"  Monitoring overhead: {m2_monitor:.3f}s")
print("=" * 70)

METHOD 1: Incremental Update (ingestion + gold build)
  Time:                145.70s
  Storage overhead:    +25.856 MB
  Monitoring overhead: 0.013s

METHOD 2: Refresh Product Tables
  Time:                13.36s
  Storage overhead:    +0.000 MB
  Monitoring overhead: 0.004s


In [9]:
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Application ID:", spark.sparkContext.applicationId)

Spark UI: http://192.168.1.8:4043
Application ID: local-1790279629641


In [10]:
import pandas as pd
from IPython.display import display
from src.ingestion.metadata import read_ingestion_runs
from src.monitoring.pipeline_monitor import read_pipeline_runs

pd.set_option("display.max_colwidth", 100)

ingestion_df = read_ingestion_runs()
pipeline_df = read_pipeline_runs()

print("=" * 70)
print("INGESTION RUNS")
print("=" * 70)
if ingestion_df is not None and not ingestion_df.empty:
    display(ingestion_df.sort_values("started_at", ascending=False))
else:
    print("No ingestion runs logged yet.")

print("\n" + "=" * 70)
print("PIPELINE RUNS (gold_build / product_refresh)")
print("=" * 70)
if pipeline_df is not None and not pipeline_df.empty:
    display(pipeline_df.sort_values("started_at", ascending=False))
else:
    print("No pipeline runs logged yet.")

INGESTION RUNS


,run_id,dataset_name,source_file,format,schema_version,status,records_read,records_bronze,records_valid_silver,records_quarantined,records_duplicate_ignored,validation_failures,storage_overhead_bytes,error_message,started_at,ended_at,execution_time_seconds
1,a758e492-af60-4e6a-a457-f052d5ee5346,air_quality,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,csv,1.0,SUCCESS,500,500,474,0,26,{},56523,,2026-09-24T19:50:03.354797+00:00,2026-09-24T19:52:09.552780+00:00,126.209
2,f1dac5ec-2056-4807-b8e3-25060a0cd30c,weather,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,csv,1.0,SUCCESS,168,168,168,0,0,{},18212,,2026-09-24T19:50:01.716593+00:00,2026-09-24T19:50:03.343284+00:00,1.627
3,cf1800da-d6b0-4e56-bee4-938050f79e9f,tripdata,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,parquet,1.0,SUCCESS,343583,343583,293670,2,49911,"{""duplicate_pk"": 2}",23738253,,2026-09-24T19:48:52.638312+00:00,2026-09-24T19:50:01.699566+00:00,69.060
4,1def6b57-cfe0-4210-84ff-6565b0d08ca7,tripdata,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,parquet,1.0,SUCCESS,3582628,3582628,3582626,2,0,"{""invalid_numeric:pickup_year"": 2}",351677747,,2026-09-24T19:42:29.478394+00:00,2026-09-24T19:44:05.925615+00:00,96.446
5,c23a579e-9475-427a-aae7-57e951bea56c,tripdata,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,parquet,1.0,SUCCESS,3007526,3007526,3007522,4,0,"{""duplicate_pk"": 2, ""invalid_numeric:pickup_year"": 2}",293534990,,2026-09-24T19:41:11.643260+00:00,2026-09-24T19:42:29.464668+00:00,77.820
6,a8e6d0a8-9569-4cf0-bb21-5c6af44872e2,tripdata,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,parquet,1.0,SUCCESS,2964624,2964624,2964619,5,0,"{""invalid_numeric:pickup_year"": 5}",289911132,,2026-09-24T19:40:19.654329+00:00,2026-09-24T19:41:11.634905+00:00,51.980
7,b803155a-db36-4a90-b8b1-f9e4762aa927,air_quality,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,csv,1.0,SUCCESS,8139551,8139551,8131572,7979,0,"{""invalid_numeric:pm25_concentration"": 7979}",65373078,,2026-09-24T19:39:04.374391+00:00,2026-09-24T19:40:19.645710+00:00,75.270
8,17582edd-4eec-4572-988b-964a5fc503cc,weather,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,csv,1.0,SUCCESS,8784,8784,8784,0,0,{},148325,,2026-09-24T19:39:02.498115+00:00,2026-09-24T19:39:04.366393+00:00,1.868
9,66b035ef-2ec6-419b-9645-8250bcfb1038,taxi_zone_lookup,/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/d...,csv,1.0,SUCCESS,265,265,265,0,0,{},10892,,2026-09-24T19:38:59.613833+00:00,2026-09-24T19:39:02.493787+00:00,2.880
0,9db88254-9d9c-4d63-a4ad-0ad36591b27a,benchmark,benchmark,parquet,v1.0,SUCCESS,0,0,0,0,0,,0,,,,0.000



PIPELINE RUNS (gold_build / product_refresh)


,run_id,pipeline_name,dataset_name,schema_version,status,records_processed,records_inserted,records_rejected,validation_failures,storage_overhead_bytes,error_message,started_at,ended_at,execution_time_seconds
2,4c79259f-84db-4039-9c8f-6240444f0745,product_refresh,data_products,v1.0,SUCCESS,293670,363,0,{},0,,2026-09-24T20:03:27.963196+00:00,2026-09-24T20:03:41.326059+00:00,13.363
3,8ce1322e-3471-4a2e-aaca-4fd0bfa7d8f6,gold_build,integrated_taxi_trips,v1.0,SUCCESS,293670,293670,0,{},25799484,,2026-09-24T19:53:53.509145+00:00,2026-09-24T19:54:13.000466+00:00,19.490
0,26af596c-723d-414b-b3f5-8fb57a0a7d30,benchmark,benchmark,v1.0,SUCCESS,0,0,0,{},0,,,,0.000
1,9cc6cec9-d2dd-4b4a-a310-42bb3cd433b1,benchmark,benchmark,v1.0,SUCCESS,0,0,0,{},0,,,,0.000


In [11]:
import pandas as pd
from IPython.display import display
from src.ingestion.metadata import read_ingestion_runs
from src.monitoring.pipeline_monitor import read_pipeline_runs

pd.set_option("display.max_colwidth", 60)

ingestion_df = read_ingestion_runs()
pipeline_df = read_pipeline_runs()

print("=" * 70)
print("TASK 3 REQUIREMENT CHECK")
print("=" * 70)

required_fields = {
    "pipeline execution time": "execution_time_seconds",
    "processed records":       "records_read",      
    "inserted records":        "records_valid_silver",  
    "rejected records":        "records_quarantined",  
    "schema version":          "schema_version",
    "validation failures":     "validation_failures",
}

print("\nIngestion runs — columns present:")
print(list(ingestion_df.columns))

print("\nPipeline runs — columns present:")
print(list(pipeline_df.columns))

print("\n--- Field-by-field check (ingestion_runs) ---")
for label, col in required_fields.items():
    present = col in ingestion_df.columns
    print(f"  {'✓' if present else '✗'} {label:<25} -> column '{col}'")

print("\n--- Sample row from ingestion_runs (most recent) ---")
display(ingestion_df.sort_values("started_at", ascending=False)[[
    "dataset_name", "schema_version", "execution_time_seconds",
    "records_read", "records_valid_silver", "records_quarantined", "validation_failures"
]].head(1))

print("\n--- Sample row from pipeline_runs (most recent) ---")
display(pipeline_df.sort_values("started_at", ascending=False)[[
    "pipeline_name", "dataset_name", "schema_version", "execution_time_seconds",
    "records_processed", "records_inserted", "records_rejected", "validation_failures"
]].head(1))

print("\n" + "=" * 70)

TASK 3 REQUIREMENT CHECK

Ingestion runs — columns present:
['run_id', 'dataset_name', 'source_file', 'format', 'schema_version', 'status', 'records_read', 'records_bronze', 'records_valid_silver', 'records_quarantined', 'records_duplicate_ignored', 'validation_failures', 'storage_overhead_bytes', 'error_message', 'started_at', 'ended_at', 'execution_time_seconds']

Pipeline runs — columns present:
['run_id', 'pipeline_name', 'dataset_name', 'schema_version', 'status', 'records_processed', 'records_inserted', 'records_rejected', 'validation_failures', 'storage_overhead_bytes', 'error_message', 'started_at', 'ended_at', 'execution_time_seconds']

--- Field-by-field check (ingestion_runs) ---
  ✓ pipeline execution time   -> column 'execution_time_seconds'
  ✓ processed records         -> column 'records_read'
  ✓ inserted records          -> column 'records_valid_silver'
  ✓ rejected records          -> column 'records_quarantined'
  ✓ schema version            -> column 'schema_version

,dataset_name,schema_version,execution_time_seconds,records_read,records_valid_silver,records_quarantined,validation_failures
1,air_quality,1.0,126.209,500,474,0,{}



--- Sample row from pipeline_runs (most recent) ---


,pipeline_name,dataset_name,schema_version,execution_time_seconds,records_processed,records_inserted,records_rejected,validation_failures
2,product_refresh,data_products,v1.0,13.363,293670,363,0,{}


In [18]:
import pandas as pd
from IPython.display import display
from src.ingestion.metadata import read_ingestion_runs

ingestion_df = read_ingestion_runs()

# 1. Clean run — SUCCESS status AND no rejections
clean = ingestion_df[
    (ingestion_df["records_quarantined"] == 0) & (ingestion_df["status"] == "SUCCESS")
].iloc[0]

# 2. Run with validation failures / rejections
with_failures = ingestion_df[
    (ingestion_df["records_quarantined"] > 0) & (ingestion_df["status"] == "SUCCESS")
].sort_values("records_quarantined", ascending=False).iloc[0]

# 3. Failed run
failed_runs = ingestion_df[ingestion_df["status"] == "FAILED"]
failed = failed_runs.iloc[0] if not failed_runs.empty else None

rows = []
for label, row in [
    ("Clean run", clean),
    ("Run with validation failures", with_failures),
    ("Failed run", failed),
]:
    if row is None:
        rows.append({
            "Scenario": label, "Dataset": "-", "Status": "no FAILED run logged",
            "Execution Time (s)": "-", "Processed": "-", "Inserted": "-",
            "Rejected": "-", "Schema Version": "-", "Validation Failures": "-",
            "Error Message": "-",
        })
    else:
        rows.append({
            "Scenario": label,
            "Dataset": row["dataset_name"],
            "Status": row["status"],
            "Execution Time (s)": row["execution_time_seconds"],
            "Processed": row["records_read"],
            "Inserted": row["records_valid_silver"],
            "Rejected": row["records_quarantined"],
            "Schema Version": row["schema_version"],
            "Validation Failures": row["validation_failures"],
            "Error Message": row["error_message"] or "-",
        })

summary_table = pd.DataFrame(rows)
display(summary_table)

,Scenario,Dataset,Status,Execution Time (s),Processed,Inserted,Rejected,Schema Version,Validation Failures,Error Message
0,Clean run,benchmark,SUCCESS,0.000,0,0,0,v1.0,,-
1,Run with validation failures,air_quality,SUCCESS,75.270,8139551,8131572,7979,1.0,"{""invalid_numeric:pm25_concentration"": 7979}",-
2,Failed run,tripdata,FAILED,0.062,0,0,0,1.0,,[PATH_NOT_FOUND] Path does not exist: file:/Users/thilin...
